# Continuous State Discretization with a VQ-VAE (MuJoCo)

Trains a VQ-VAE that maps the continuous states of a MuJoCo environment to discrete codebook indices, then studies
which transitions stay in the same discrete state (self-loops) and which actions cause them.

## 1. Imports

In [ ]:
import os
import random

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pandas.plotting import parallel_coordinates
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from torch.distributions.normal import Normal
from torch.utils.data import DataLoader, TensorDataset

## 2. Configuration and hyperparameters

In [ ]:
SEED = 42
ENV_NAME = 'Hopper-v4'

# Datasets
RANDOM_POLICY_SAMPLES = 25000           # transitions from a uniform random policy (quick baseline, only summarised)
POLICY_SAMPLES = 200000                 # transitions from the policy network

# VQ-VAE state discretizer
NUM_EMBEDDINGS = 256                    # number of discrete states (codebook size K)
EMBEDDING_DIM = 4                       # dimension of each codebook vector
VQVAE_HIDDEN_DIM = 64
COMMITMENT_COST = 0.25
VQVAE_BATCH_SIZE = 256
VQVAE_EPOCHS = 50
VQVAE_LEARNING_RATE = 1e-3

# Codebook-size sweep and the dynamics-aware sketch (section 8)
SWEEP_CODEBOOK_SIZES = [64, 128, 256, 512, 1024]
SWEEP_EPOCHS = 40
SWEEP_SEED = 0
TRANSITION_LOSS_WEIGHT = 1.0            # weight of the next-code prediction loss in the dynamics-aware sketch

# Self-loop precision per cluster (section 9)
PRECISION_THRESHOLD = 0.70              # a cluster gives a valid mask when the self-loop precision reaches this value
MIN_CLUSTER_SAMPLES = 20                # smaller clusters are not evaluated
NUM_CLUSTERS_TO_PLOT = 4
PCA_OUTPUT_DIR = "plots_pca"
TSNE_OUTPUT_DIR = "plots_tsne"
PARALLEL_COORDINATES_SAMPLES = 300      # transitions drawn in the parallel-coordinates plot

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3. Environment and data collection

In [ ]:
env = gym.make(ENV_NAME)
print(f"Successfully initialized Gymnasium environment: {ENV_NAME}")

state_space = env.observation_space
action_space = env.action_space
action_low = env.action_space.low
action_high = env.action_space.high
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

print(f"State Space : {state_space}, Action Space : {action_space}")
print(f"Action Limits: Min = {action_low}, Max = {action_high}")

Successfully initialized Gymnasium environment: Hopper-v4
State Space : Box(-inf, inf, (11,), float64), Action Space : Box(-1.0, 1.0, (3,), float32)
Action Limits: Min = [-1. -1. -1.], Max = [1. 1. 1.]


In [ ]:
# Quick baseline: transitions from a uniform random policy.  They are only summarised here;
# the dataset collected with the policy network (next cells) replaces them.
states = []
actions = []
next_states = []

obs, _ = env.reset(seed=SEED)
for _ in range(RANDOM_POLICY_SAMPLES):
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, _ = env.step(action)

    states.append(obs)
    actions.append(action)
    next_states.append(next_obs)

    if terminated or truncated:
        obs, _ = env.reset()
    else:
        obs = next_obs

states_np = np.array(states, dtype=np.float32)
state_mean = torch.tensor(states_np.mean(axis=0), dtype=torch.float32)
state_std = torch.tensor(states_np.std(axis=0) + 1e-6, dtype=torch.float32)

print(f"Dataset collected: {len(states_np)} samples.")
print(f"State Means: {state_mean.numpy()}")
print(f"State Stds:  {state_std.numpy()}")

Dataset collected: 25000 samples.
State Means: [ 1.2240664  -0.06043873 -0.0431489  -0.05778847  0.04066097 -0.15154015
 -0.23735347 -1.1115062  -0.7926648  -0.8189063   0.46764743]
State Stds:  [0.02410142 0.06326925 0.06425974 0.09151178 0.1449656  0.40609643
 0.29261723 1.3932205  1.4512552  1.4545401  1.7183121 ]


In [ ]:
class GaussianPolicy(nn.Module):
    """Gaussian policy network: state -> mean of every action dimension, plus a learned log-std."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.mean_network = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, action_dim),
        )
        self.log_std = nn.Parameter(torch.zeros(1, action_dim))

    def get_action(self, state):
        action_mean = self.mean_network(state)
        action_log_std = self.log_std.expand_as(action_mean)
        action_std = torch.exp(action_log_std)
        action_distribution = Normal(action_mean, action_std)
        action = action_distribution.sample()
        return action.detach().cpu().numpy()


# No checkpoint is loaded in this notebook: the policy acts with its freshly initialised weights.
policy = GaussianPolicy(state_dim, action_dim).to(device)
policy.eval()

In [ ]:
# Transitions from the policy network.  They replace the random-policy data.
states = []
actions = []
next_states = []

obs, _ = env.reset(seed=SEED)
for _ in range(POLICY_SAMPLES):
    obs_tensor = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        action = policy.get_action(obs_tensor)[0]
    action = np.clip(action, env.action_space.low, env.action_space.high)
    next_obs, reward, terminated, truncated, _ = env.step(action)

    states.append(obs)
    actions.append(action)
    next_states.append(next_obs)

    if terminated or truncated:
        obs, _ = env.reset()
    else:
        obs = next_obs

states_np = np.array(states, dtype=np.float32)                  # (N, state_dim)
actions_np = np.array(actions, dtype=np.float32)                # (N, action_dim)
next_states_np = np.array(next_states, dtype=np.float32)        # (N, state_dim)

# States are normalised with the mean and standard deviation of this dataset
state_mean = torch.tensor(states_np.mean(axis=0), dtype=torch.float32)
state_std = torch.tensor(states_np.std(axis=0) + 1e-6, dtype=torch.float32)
states_normalized = (torch.tensor(states_np, dtype=torch.float32) - state_mean) / state_std
next_states_normalized = (torch.tensor(next_states_np, dtype=torch.float32) - state_mean) / state_std

dataloader = DataLoader(TensorDataset(states_normalized), batch_size=VQVAE_BATCH_SIZE, shuffle=True)

print(f"Dataset collected with RL policy: {len(states_np)} samples.")
print(f"State Means: {state_mean.numpy()}")
print(f"State Stds:  {state_std.numpy()}")

## 4. VQ-VAE model

In [ ]:
class VectorQuantizer(nn.Module):
    """Replaces each encoder output by its nearest codebook vector; the index of that vector is the discrete state."""

    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)     # the codebook
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, inputs):
        # inputs: encoder outputs z_e, shape (batch, embedding_dim)
        # Squared distance of every input to every codebook vector, shape (batch, num_embeddings):
        # ||z - e||^2 = ||z||^2 + ||e||^2 - 2 z.e
        distances = (torch.sum(inputs**2, dim=1, keepdim=True)
                     + torch.sum(self.embedding.weight**2, dim=1)
                     - 2 * torch.matmul(inputs, self.embedding.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)       # index of the nearest codebook vector, (batch,)
        quantized = self.embedding(encoding_indices)            # (batch, embedding_dim)

        commitment_loss = F.mse_loss(quantized.detach(), inputs)    # pulls the encoder output towards its (fixed) code
        codebook_loss = F.mse_loss(quantized, inputs.detach())      # pulls the code towards the (fixed) encoder output
        loss = codebook_loss + self.commitment_cost * commitment_loss

        # Straight-through estimator: the forward pass uses the quantized vector,
        # the backward pass copies its gradient to the encoder output unchanged.
        quantized = inputs + (quantized - inputs).detach()

        # Perplexity = exp(entropy of the code usage in this batch): the effective number of codes in use
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()
        code_usage = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(code_usage * torch.log(code_usage + 1e-10)))

        return quantized, loss, encoding_indices, perplexity

In [ ]:
class VQVAEStateDiscretizer(nn.Module):
    """Autoencoder with a vector-quantizer bottleneck: state -> encoder -> nearest code -> decoder -> reconstructed state."""

    def __init__(self, state_dim, hidden_dim, embedding_dim, num_embeddings, commitment_cost):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )
        self.vq_layer = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, state_dim)
        )

    def forward(self, state):
        z_e = self.encoder(state)
        z_q, vq_loss, encoding_indices, perplexity = self.vq_layer(z_e)
        state_recon = self.decoder(z_q)
        return state_recon, vq_loss, encoding_indices, perplexity

    def encode_to_discrete(self, state):
        """Codebook index of each state, shape (batch,)."""
        z_e = self.encoder(state)
        _, _, encoding_indices, _ = self.vq_layer(z_e)
        return encoding_indices


vqvae_model = VQVAEStateDiscretizer(
    state_dim=state_dim,
    hidden_dim=VQVAE_HIDDEN_DIM,
    embedding_dim=EMBEDDING_DIM,
    num_embeddings=NUM_EMBEDDINGS,
    commitment_cost=COMMITMENT_COST
).to(device)

print(vqvae_model)

## 5. Training the VQ-VAE

In [ ]:
vqvae_optimizer = torch.optim.Adam(vqvae_model.parameters(), lr=VQVAE_LEARNING_RATE)

recon_loss_history = []
vq_loss_history = []
perplexity_history = []

vqvae_model.train()
for epoch in range(1, VQVAE_EPOCHS + 1):
    total_loss_epoch = 0.0
    total_recon_epoch = 0.0
    total_vq_epoch = 0.0
    total_perplexity_epoch = 0.0

    for (batch_states,) in dataloader:
        batch_states = batch_states.to(device)

        vqvae_optimizer.zero_grad()
        state_recon, vq_loss, _, perplexity = vqvae_model(batch_states)
        recon_loss = F.mse_loss(state_recon, batch_states)
        loss = recon_loss + vq_loss
        loss.backward()
        vqvae_optimizer.step()

        total_loss_epoch += loss.item()
        total_recon_epoch += recon_loss.item()
        total_vq_epoch += vq_loss.item()
        total_perplexity_epoch += perplexity.item()

    num_batches = len(dataloader)
    avg_loss = total_loss_epoch / num_batches
    avg_recon = total_recon_epoch / num_batches
    avg_vq = total_vq_epoch / num_batches
    avg_perplexity = total_perplexity_epoch / num_batches

    recon_loss_history.append(avg_recon)
    vq_loss_history.append(avg_vq)
    perplexity_history.append(avg_perplexity)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{VQVAE_EPOCHS}] | Total Loss: {avg_loss:.5f} | Recon MSE: {avg_recon:.5f} | VQ Loss: {avg_vq:.5f} | Codebook Perplexity: {avg_perplexity:.2f}/{NUM_EMBEDDINGS}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(recon_loss_history, label='Reconstruction MSE Loss', color='tab:blue')
axes[0].plot(vq_loss_history, label='VQ Loss', color='tab:orange')
axes[0].set_title('VQ-VAE Loss Convergence')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(perplexity_history, label='Active Codebook Utilization (Perplexity)', color='tab:green')
axes[1].axhline(y=NUM_EMBEDDINGS, color='r', linestyle='--', label=f'Max Codebook Capacity ({NUM_EMBEDDINGS})')
axes[1].set_title('Codebook Perplexity over Training')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

## 6. Discrete states: PCA projection

In [ ]:
# Discrete state (codebook index) of every state and of every next state
vqvae_model.eval()
with torch.no_grad():
    state_cluster_ids = vqvae_model.encode_to_discrete(states_normalized.to(device)).cpu().numpy()              # (N,)
    next_state_cluster_ids = vqvae_model.encode_to_discrete(next_states_normalized.to(device)).cpu().numpy()    # (N,)

print(f"Unique discrete state clusters used: {len(np.unique(state_cluster_ids))} out of {NUM_EMBEDDINGS}")

In [ ]:
if states_np.shape[1] == 1:
    # a one-dimensional state has nothing to project
    pca_states = np.column_stack([states_np[:, 0], np.zeros_like(states_np[:, 0])])
    variance_explained_pct = [100.0, 0.0]
else:
    # PCA of the standardised states through an SVD; the states are projected on the top two components
    pca_mean = np.mean(states_np, axis=0)
    pca_std = np.std(states_np, axis=0)
    pca_std[pca_std == 0] = 1.0             # avoid a division by zero for constant dimensions
    states_standardized = (states_np - pca_mean) / pca_std
    _, singular_values, principal_axes = np.linalg.svd(states_standardized, full_matrices=False)
    pca_states = states_standardized @ principal_axes[:2].T            # (N, 2)

    explained_variance = (singular_values**2) / (len(states_np) - 1)
    variance_explained_pct = (explained_variance[:2] / np.sum(explained_variance)) * 100

plt.figure(figsize=(10, 7))
cluster_scatter = plt.scatter(pca_states[:, 0], pca_states[:, 1], c=state_cluster_ids, cmap='jet', alpha=0.6, s=12)
colorbar = plt.colorbar(cluster_scatter)
colorbar.set_label('Discrete State Index (Codebook ID)', fontsize=11)

plt.xlabel(f'PCA Component 1 ({variance_explained_pct[0]:.1f}% Variance Explained)', fontsize=11)
plt.ylabel(f'PCA Component 2 ({variance_explained_pct[1]:.1f}% Variance Explained)', fontsize=11)
plt.title(f'VQ-VAE State Space Discretization (PCA Projection) for {ENV_NAME}\nTotal Variance Explained: {sum(variance_explained_pct[:2]):.1f}%', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Self-loop transitions (action-masking analysis)

In [ ]:
# A transition is a self-loop when s_t and s_{t+1} fall in the same discrete state
total_transitions = len(state_cluster_ids)
is_self_loop = (state_cluster_ids == next_state_cluster_ids)

num_self_loops = np.sum(is_self_loop)
num_state_changes = total_transitions - num_self_loops
self_loop_pct = (num_self_loops / total_transitions) * 100
state_change_pct = (num_state_changes / total_transitions) * 100

print("="*60)
print(f"REPLAY BUFFER DISCRETE TRANSITION ANALYSIS ({ENV_NAME})")
print("="*60)
print(f"Total Trajectory Transitions (s_t -> s_{{t+1}}): {total_transitions:,}")
print(f"Self-Loop Transitions (z(s_t) == z(s_{{t+1}})): {num_self_loops:,} ({self_loop_pct:.2f}%)")
print(f"State-Changing Transitions (z(s_t) != z(s_{{t+1}})): {num_state_changes:,} ({state_change_pct:.2f}%)")
print("="*60)

# Per cluster: number of transitions that start in it, and the share of them that are self-loops
cluster_total_counts = np.zeros(NUM_EMBEDDINGS, dtype=int)
cluster_self_loop_counts = np.zeros(NUM_EMBEDDINGS, dtype=int)
for cluster_id in range(NUM_EMBEDDINGS):
    in_cluster = (state_cluster_ids == cluster_id)
    cluster_total_counts[cluster_id] = np.sum(in_cluster)
    if cluster_total_counts[cluster_id] > 0:
        cluster_self_loop_counts[cluster_id] = np.sum(is_self_loop[in_cluster])

active_clusters = cluster_total_counts > 0
self_loop_ratios = np.zeros(NUM_EMBEDDINGS)
self_loop_ratios[active_clusters] = (cluster_self_loop_counts[active_clusters] / cluster_total_counts[active_clusters]) * 100

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: self-loops versus state changes over the whole dataset
axes[0].bar(['Self-Loops (z(s_t) == z(s_t+1))', 'State Changes (z(s_t) != z(s_t+1))'],
            [num_self_loops, num_state_changes], color=['tab:green', 'tab:blue'], alpha=0.8)
axes[0].set_ylabel('Number of Transitions')
axes[0].set_title(f'Replay Buffer Transitions ({ENV_NAME})')
axes[0].grid(True, linestyle='--', alpha=0.5)
for bar in axes[0].patches:
    axes[0].annotate(f"{int(bar.get_height()):,}\n({bar.get_height()/total_transitions*100:.1f}%)",
                     (bar.get_x() + bar.get_width() / 2, bar.get_height() / 2),
                     ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# Right: self-loop rate of each cluster that occurs in the data
active_cluster_ids = np.where(active_clusters)[0]
axes[1].bar(active_cluster_ids, self_loop_ratios[active_cluster_ids], color='tab:purple', alpha=0.7)
axes[1].set_xlabel('Discrete Cluster ID')
axes[1].set_ylabel('Self-Loop Transition Rate (%)')
axes[1].set_title('State Retention Ratio per Discrete Codebook Cluster')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 8. Masking diagnostics

These cells need `masking_diagnostics.py` next to the notebook.

In [ ]:
import masking_diagnostics as md

# Codebook usage, spread of the states inside each cell, and the transition structure between cells
z_e, codebook = md.encoder_outputs(vqvae_model, states_normalized, device)

md.codebook_usage_report(state_cluster_ids, NUM_EMBEDDINGS)
md.within_cluster_spread(states_normalized.numpy(), state_cluster_ids, NUM_EMBEDDINGS)
md.transition_matrix_plot(state_cluster_ids, next_state_cluster_ids, NUM_EMBEDDINGS);

In [ ]:
# Can a classifier tell self-loops from state changes inside a cell: from the action alone,
# and from the action plus the position of the state inside its cell?
separability_action_only = md.separability_report(actions_np, state_cluster_ids, is_self_loop, title="action only")
position_features = md.cell_position_features(z_e, codebook, state_cluster_ids)
separability_action_and_position = md.separability_report(actions_np, state_cluster_ids, is_self_loop,
                                                          extra_features=position_features, title="action + position-in-cell")
md.compare_reports({"action only": separability_action_only, "action + position": separability_action_and_position});

In [ ]:
# Is a self-loop explained by the distance of the state to its cell boundary rather than by the action?
md.boundary_margin_analysis(z_e, codebook, state_cluster_ids, is_self_loop, actions_np)
md.pca_cells_plot(states_np, state_cluster_ids, sep_df=separability_action_only, K=NUM_EMBEDDINGS)

### Codebook-size sweep: are more (or fewer) than `NUM_EMBEDDINGS` codes needed?

In [ ]:
# One VQ-VAE is trained per codebook size and scored with the same diagnostics.
sweep_results = []
for num_embeddings in SWEEP_CODEBOOK_SIZES:
    torch.manual_seed(SWEEP_SEED)
    sweep_model = VQVAEStateDiscretizer(state_dim, VQVAE_HIDDEN_DIM, EMBEDDING_DIM, num_embeddings, COMMITMENT_COST).to(device)

    # Starting the codebook at the encoder outputs of randomly chosen states (as k-means does) improves its utilisation a lot.
    with torch.no_grad():
        initial_state_indices = torch.randperm(len(states_normalized))[:num_embeddings]
        sweep_model.vq_layer.embedding.weight.copy_(sweep_model.encoder(states_normalized[initial_state_indices].to(device)))

    sweep_optimizer = torch.optim.Adam(sweep_model.parameters(), lr=VQVAE_LEARNING_RATE)
    sweep_model.train()
    for _ in range(SWEEP_EPOCHS):
        for (batch_states,) in dataloader:
            batch_states = batch_states.to(device)
            state_recon, vq_loss, _, _ = sweep_model(batch_states)
            loss = F.mse_loss(state_recon, batch_states) + vq_loss
            sweep_optimizer.zero_grad()
            loss.backward()
            sweep_optimizer.step()

    sweep_model.eval()
    with torch.no_grad():
        sweep_cluster_ids = sweep_model.encode_to_discrete(states_normalized.to(device)).cpu().numpy()
        sweep_next_cluster_ids = sweep_model.encode_to_discrete(next_states_normalized.to(device)).cpu().numpy()

    # Only the 100 largest cells are scored for separability, which keeps the sweep fast.
    sweep_row = md.evaluate_discretization(sweep_cluster_ids, sweep_next_cluster_ids, actions_np, states_normalized.numpy(),
                                           num_embeddings, cv=3, max_cells_for_sep=100)
    print({key: (round(value, 3) if isinstance(value, float) else value) for key, value in sweep_row.items()})
    sweep_results.append(sweep_row)

md.plot_k_sweep(sweep_results)

### Sketch: dynamics-aware discretizer (defined here, not run in this notebook)

A head predicts the *next* code from $(z_q(s_t), a_t)$. This pushes the encoder to carve cells inside which "does action $a$ leave the cell?" has a consistent answer, which is exactly the property the mask $M(k, a)$ needs; plain reconstruction has no reason to provide it. After training, re-run the diagnostics above on the returned model and compare the weighted AUROC and the share of transitions in separable cells against the plain VQ-VAE.

In [ ]:
class TransitionHead(nn.Module):
    """Predicts the code of the next state: (z_q(s_t), a_t) -> one logit per code."""

    def __init__(self, embedding_dim, action_dim, num_embeddings, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim + action_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, num_embeddings))

    def forward(self, z_q, action):
        return self.net(torch.cat([z_q, action], dim=1))


def train_dynamics_aware(num_embeddings):
    """Trains a VQ-VAE together with a TransitionHead on (s_t, a_t, s_{t+1}); returns (model, transition_head)."""
    torch.manual_seed(SWEEP_SEED)
    model = VQVAEStateDiscretizer(state_dim, VQVAE_HIDDEN_DIM, EMBEDDING_DIM, num_embeddings, COMMITMENT_COST).to(device)
    transition_head = TransitionHead(EMBEDDING_DIM, action_dim, num_embeddings).to(device)
    transitions = TensorDataset(states_normalized, torch.tensor(actions_np), next_states_normalized)
    transition_loader = DataLoader(transitions, batch_size=VQVAE_BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(list(model.parameters()) + list(transition_head.parameters()), lr=VQVAE_LEARNING_RATE)
    for _ in range(SWEEP_EPOCHS):
        for batch_states, batch_actions, batch_next_states in transition_loader:
            batch_states = batch_states.to(device)
            batch_actions = batch_actions.to(device)
            batch_next_states = batch_next_states.to(device)
            z_e = model.encoder(batch_states)
            z_q, vq_loss, _, _ = model.vq_layer(z_e)
            state_recon = model.decoder(z_q)
            with torch.no_grad():       # target: the code of the next state
                next_codes = model.encode_to_discrete(batch_next_states)
            transition_loss = F.cross_entropy(transition_head(z_q, batch_actions), next_codes)
            loss = F.mse_loss(state_recon, batch_states) + vq_loss + TRANSITION_LOSS_WEIGHT * transition_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, transition_head

## 9. Which actions cause self-loops?

In [ ]:
# For each cluster: can a depth-5 decision tree predict from the action alone whether a transition is a self-loop?
# The precision of the self-loop class on held-out transitions decides whether the cluster gives a valid mask.
clusters_by_size = np.argsort(cluster_total_counts)[::-1]

print("=" * 85)
print(f"EVALUATING SELF-LOOP PRECISION ACROSS CLUSTERS (Target Self-Loop Precision >= {PRECISION_THRESHOLD*100:.0f}%)")
print("=" * 85)

cluster_metrics = {}
for cluster_id in clusters_by_size:
    in_cluster = (state_cluster_ids == cluster_id)
    cluster_actions = actions_np[in_cluster]
    self_loop_labels = is_self_loop[in_cluster].astype(int)         # 1 = self-loop, 0 = state change
    num_samples = len(cluster_actions)
    if num_samples < MIN_CLUSTER_SAMPLES:
        continue

    if len(np.unique(self_loop_labels)) < 2:
        # only one class occurs in this cluster, so there is nothing to separate
        only_self_loops = self_loop_labels[0] == 1
        precision = 1.0 if only_self_loops else 0.0
        recall = 1.0 if only_self_loops else 0.0
    else:
        train_actions, test_actions, train_labels, test_labels = train_test_split(
            cluster_actions, self_loop_labels, test_size=0.3, random_state=SEED, stratify=self_loop_labels
        )
        tree = DecisionTreeClassifier(max_depth=5, random_state=SEED, class_weight="balanced")
        tree.fit(train_actions, train_labels)
        predicted_labels = tree.predict(test_actions)
        precision = precision_score(test_labels, predicted_labels, pos_label=1, zero_division=0)
        recall = recall_score(test_labels, predicted_labels, pos_label=1, zero_division=0)

    is_valid_mask = precision >= PRECISION_THRESHOLD
    status = "SEPARABLE (Valid Mask)" if is_valid_mask else "REJECTED (Low Precision)"
    cluster_metrics[cluster_id] = {"precision": precision, "recall": recall, "is_valid": is_valid_mask}

    print(f"Cluster ID {cluster_id:3d} | Samples: {num_samples:5d} | "
          f"Self-Loop Ratio: {np.mean(self_loop_labels)*100:5.1f}% | "
          f"Self-Loop Precision: {precision*100:6.2f}% | "
          f"Recall: {recall*100:6.2f}% | Status: {status}")
print("=" * 85)

# The largest evaluated clusters are drawn in the PCA and t-SNE figures below
clusters_to_plot = [cluster_id for cluster_id in clusters_by_size if cluster_id in cluster_metrics][:NUM_CLUSTERS_TO_PLOT]

In [ ]:
# Actions of each cluster projected with PCA: 2-D with the regions of a depth-4 decision tree (left), 3-D (right)
os.makedirs(PCA_OUTPUT_DIR, exist_ok=True)
fig = plt.figure(figsize=(16, 5.5 * len(clusters_to_plot)), dpi=300)

for row, cluster_id in enumerate(clusters_to_plot):
    metrics = cluster_metrics[cluster_id]
    in_cluster = (state_cluster_ids == cluster_id)
    cluster_actions = actions_np[in_cluster]
    cluster_self_loops = is_self_loop[in_cluster]
    status = "SEPARABLE (Valid Mask)" if metrics["is_valid"] else "REJECTED (Low Precision)"
    status_color = "darkgreen" if metrics["is_valid"] else "crimson"

    ax_2d = fig.add_subplot(len(clusters_to_plot), 2, 2 * row + 1)
    if action_dim > 1:
        actions_2d = PCA(n_components=min(2, action_dim)).fit_transform(cluster_actions)
    else:
        actions_2d = np.column_stack((cluster_actions, np.zeros_like(cluster_actions)))

    if len(np.unique(cluster_self_loops)) > 1:
        boundary_tree = DecisionTreeClassifier(max_depth=4, random_state=SEED, class_weight="balanced")
        boundary_tree.fit(actions_2d, cluster_self_loops.astype(int))
        x_min, x_max = actions_2d[:, 0].min() - 0.2, actions_2d[:, 0].max() + 0.2
        y_min, y_max = actions_2d[:, 1].min() - 0.2, actions_2d[:, 1].max() + 0.2
        grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        predicted_class = boundary_tree.predict(np.c_[grid_x.ravel(), grid_y.ravel()]).reshape(grid_x.shape)
        ax_2d.contourf(grid_x, grid_y, predicted_class, alpha=0.15, cmap=plt.cm.RdYlGn_r)

    ax_2d.scatter(actions_2d[cluster_self_loops, 0], actions_2d[cluster_self_loops, 1],
                  c="tab:red", label="Self-Loop (Penalized)", alpha=0.7, s=35, marker="x")
    ax_2d.scatter(actions_2d[~cluster_self_loops, 0], actions_2d[~cluster_self_loops, 1],
                  c="tab:green", label="State Change (Allowed)", alpha=0.5, s=25, marker="o")
    ax_2d.set_title(f"Cluster ID {cluster_id} — 2D PCA Decision Boundary", fontsize=11, fontweight="bold")
    ax_2d.set_xlabel("PCA Dim 1", fontsize=9)
    ax_2d.set_ylabel("PCA Dim 2", fontsize=9)
    ax_2d.grid(True, linestyle="--", alpha=0.4)
    ax_2d.legend(loc="best", fontsize=8)
    ax_2d.text(0.02, 0.90, f"Self-Loop Precision: {metrics['precision']*100:.1f}%\nRecall: {metrics['recall']*100:.1f}%\nStatus: {status}",
               transform=ax_2d.transAxes, fontsize=9, fontweight="bold",
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=status_color, alpha=0.9))

    ax_3d = fig.add_subplot(len(clusters_to_plot), 2, 2 * row + 2, projection='3d')
    if action_dim >= 3:
        actions_3d = PCA(n_components=min(3, action_dim)).fit_transform(cluster_actions)
    else:
        actions_3d = np.hstack((cluster_actions, np.zeros((len(cluster_actions), 3 - action_dim))))
    ax_3d.scatter(actions_3d[cluster_self_loops, 0], actions_3d[cluster_self_loops, 1], actions_3d[cluster_self_loops, 2],
                  c="tab:red", label="Self-Loop (Penalized)", alpha=0.7, s=35, marker="x")
    ax_3d.scatter(actions_3d[~cluster_self_loops, 0], actions_3d[~cluster_self_loops, 1], actions_3d[~cluster_self_loops, 2],
                  c="tab:green", label="State Change (Allowed)", alpha=0.5, s=25, marker="o")
    ax_3d.set_title(f"Cluster ID {cluster_id} — 3D PCA Projection", fontsize=11, fontweight="bold")
    ax_3d.legend(loc="best", fontsize=8)

plt.tight_layout()
fig.suptitle(f"Self-Loop Precision Threshold Verification ({ENV_NAME})", y=1.01, fontsize=13, fontweight="bold")

save_path = os.path.join(PCA_OUTPUT_DIR, f"pca_self_loop_precision_check_{ENV_NAME}.png")
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"[Saved] Self-loop precision check plot saved to: {save_path}")
plt.show()

In [ ]:
# The same clusters projected with t-SNE instead of PCA
os.makedirs(TSNE_OUTPUT_DIR, exist_ok=True)
fig = plt.figure(figsize=(16, 5.5 * len(clusters_to_plot)), dpi=300)

for row, cluster_id in enumerate(clusters_to_plot):
    metrics = cluster_metrics[cluster_id]
    in_cluster = (state_cluster_ids == cluster_id)
    cluster_actions = actions_np[in_cluster]
    cluster_self_loops = is_self_loop[in_cluster]
    status = "SEPARABLE (Valid Mask)" if metrics["is_valid"] else "REJECTED (Low Precision)"
    status_color = "darkgreen" if metrics["is_valid"] else "crimson"
    tsne_perplexity = min(30, max(5, (len(cluster_actions) - 1) // 3))

    ax_2d = fig.add_subplot(len(clusters_to_plot), 2, 2 * row + 1)
    if action_dim > 1:
        tsne_2d = TSNE(n_components=2, perplexity=tsne_perplexity, random_state=SEED, init='pca', learning_rate='auto')
        actions_2d = tsne_2d.fit_transform(cluster_actions)
    else:
        actions_2d = np.column_stack((cluster_actions, np.zeros_like(cluster_actions)))

    if len(np.unique(cluster_self_loops)) > 1:
        boundary_tree = DecisionTreeClassifier(max_depth=4, random_state=SEED, class_weight="balanced")
        boundary_tree.fit(actions_2d, cluster_self_loops.astype(int))
        x_min, x_max = actions_2d[:, 0].min() - 2.0, actions_2d[:, 0].max() + 2.0
        y_min, y_max = actions_2d[:, 1].min() - 2.0, actions_2d[:, 1].max() + 2.0
        grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        predicted_class = boundary_tree.predict(np.c_[grid_x.ravel(), grid_y.ravel()]).reshape(grid_x.shape)
        ax_2d.contourf(grid_x, grid_y, predicted_class, alpha=0.15, cmap=plt.cm.RdYlGn_r)

    ax_2d.scatter(actions_2d[cluster_self_loops, 0], actions_2d[cluster_self_loops, 1],
                  c="tab:red", label="Self-Loop (Penalized)", alpha=0.7, s=35, marker="x")
    ax_2d.scatter(actions_2d[~cluster_self_loops, 0], actions_2d[~cluster_self_loops, 1],
                  c="tab:green", label="State Change (Allowed)", alpha=0.5, s=25, marker="o")
    ax_2d.set_title(f"Cluster ID {cluster_id} — 2D t-SNE Boundary", fontsize=11, fontweight="bold")
    ax_2d.set_xlabel("t-SNE Dim 1", fontsize=9)
    ax_2d.set_ylabel("t-SNE Dim 2", fontsize=9)
    ax_2d.grid(True, linestyle="--", alpha=0.4)
    ax_2d.legend(loc="best", fontsize=8)
    ax_2d.text(0.02, 0.90, f"Self-Loop Precision: {metrics['precision']*100:.1f}%\nRecall: {metrics['recall']*100:.1f}%\nStatus: {status}",
               transform=ax_2d.transAxes, fontsize=9, fontweight="bold",
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=status_color, alpha=0.9))

    ax_3d = fig.add_subplot(len(clusters_to_plot), 2, 2 * row + 2, projection='3d')
    if action_dim >= 3:
        tsne_3d = TSNE(n_components=3, perplexity=tsne_perplexity, random_state=SEED, init='pca', learning_rate='auto')
        actions_3d = tsne_3d.fit_transform(cluster_actions)
    else:
        actions_3d = np.hstack((cluster_actions, np.zeros((len(cluster_actions), 3 - action_dim))))
    ax_3d.scatter(actions_3d[cluster_self_loops, 0], actions_3d[cluster_self_loops, 1], actions_3d[cluster_self_loops, 2],
                  c="tab:red", label="Self-Loop (Penalized)", alpha=0.7, s=35, marker="x")
    ax_3d.scatter(actions_3d[~cluster_self_loops, 0], actions_3d[~cluster_self_loops, 1], actions_3d[~cluster_self_loops, 2],
                  c="tab:green", label="State Change (Allowed)", alpha=0.5, s=25, marker="o")
    ax_3d.set_title(f"Cluster ID {cluster_id} — 3D t-SNE Projection", fontsize=11, fontweight="bold")
    ax_3d.legend(loc="best", fontsize=8)

plt.tight_layout()
fig.suptitle(f"Self-Loop Precision Threshold Verification — t-SNE ({ENV_NAME})", y=1.01, fontsize=13, fontweight="bold")

save_path = os.path.join(TSNE_OUTPUT_DIR, f"tsne_self_loop_precision_check_{ENV_NAME}.png")
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"[Saved] Self-loop precision t-SNE plot saved to: {save_path}")
plt.show()

In [ ]:
# Parallel coordinates of a random subset of actions (left) and the distribution of every action dimension (right)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), dpi=300)

sample_size = min(PARALLEL_COORDINATES_SAMPLES, len(actions_np))
sample_indices = np.random.choice(len(actions_np), size=sample_size, replace=False)
sampled_actions = pd.DataFrame(actions_np[sample_indices], columns=[f"Act {i}" for i in range(action_dim)])
sampled_actions["Transition_Type"] = ["Self-Loop" if self_loop else "State-Change" for self_loop in is_self_loop[sample_indices]]

parallel_coordinates(sampled_actions, class_column="Transition_Type", color=["tab:green", "tab:red"],
                     alpha=0.4, linewidth=1.2, ax=axes[0])
axes[0].set_title("Parallel Coordinates of Continuous Action Vectors", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Action Control Value")
axes[0].grid(True, linestyle="--", alpha=0.3)

self_loop_actions_per_dim = [actions_np[is_self_loop, i] for i in range(action_dim)]
state_change_actions_per_dim = [actions_np[~is_self_loop, i] for i in range(action_dim)]
self_loop_positions = np.arange(action_dim) * 2.0
state_change_positions = np.arange(action_dim) * 2.0 + 0.7

self_loop_boxes = axes[1].boxplot(self_loop_actions_per_dim, positions=self_loop_positions, widths=0.5, patch_artist=True,
                                  boxprops=dict(facecolor="tab:green", alpha=0.6),
                                  medianprops=dict(color="black", linewidth=1.5))
state_change_boxes = axes[1].boxplot(state_change_actions_per_dim, positions=state_change_positions, widths=0.5, patch_artist=True,
                                     boxprops=dict(facecolor="tab:red", alpha=0.6),
                                     medianprops=dict(color="black", linewidth=1.5))

axes[1].set_xticks(self_loop_positions + 0.35)
axes[1].set_xticklabels([f"Dim {i}" for i in range(action_dim)])
axes[1].set_title("Action Value Distribution per Control Dimension", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Action Dimension")
axes[1].set_ylabel("Continuous Action Value")
axes[1].grid(True, linestyle="--", alpha=0.3)
axes[1].legend([self_loop_boxes["boxes"][0], state_change_boxes["boxes"][0]],
               ["Self-Loop (Retained)", "State-Change (Masked)"], loc="best")

plt.tight_layout()
plt.show()

In [ ]:
# Mean self-loop action of every cluster (left) and the polar distribution of all self-loop actions (right)
mean_self_loop_action = np.zeros((NUM_EMBEDDINGS, action_dim))
self_loops_per_cluster = np.zeros(NUM_EMBEDDINGS)
for cluster_id in range(NUM_EMBEDDINGS):
    self_loops_in_cluster = (state_cluster_ids == cluster_id) & is_self_loop
    self_loops_per_cluster[cluster_id] = np.sum(self_loops_in_cluster)
    if self_loops_per_cluster[cluster_id] > 0:
        mean_self_loop_action[cluster_id] = np.mean(actions_np[self_loops_in_cluster], axis=0)
clusters_with_self_loops = np.where(self_loops_per_cluster > 0)[0]

fig = plt.figure(figsize=(16, 6), dpi=300)

ax_heatmap = fig.add_subplot(1, 2, 1)
heatmap = ax_heatmap.imshow(mean_self_loop_action[clusters_with_self_loops], aspect="auto", cmap="coolwarm", vmin=-1.0, vmax=1.0)
ax_heatmap.set_title("Mean Continuous Action Values for Self-Loops per Cluster", fontsize=11, fontweight="bold")
ax_heatmap.set_xlabel("Action Control Dimension")
ax_heatmap.set_ylabel("Active Codebook Cluster Index")
ax_heatmap.set_xticks(range(action_dim))
ax_heatmap.set_yticks(range(len(clusters_with_self_loops)))
ax_heatmap.set_yticklabels(clusters_with_self_loops)
heatmap_colorbar = fig.colorbar(heatmap, ax=ax_heatmap)
heatmap_colorbar.set_label("Mean Action Value", fontsize=9)

# Polar view: angle and length of the vector formed by the first two action dimensions
ax_polar = fig.add_subplot(1, 2, 2, projection="polar")
self_loop_actions = actions_np[is_self_loop]
if action_dim >= 2:
    angles = np.arctan2(self_loop_actions[:, 1], self_loop_actions[:, 0])
    radii = np.linalg.norm(self_loop_actions[:, :2], axis=1)
else:
    angles = self_loop_actions[:, 0]
    radii = np.abs(self_loop_actions[:, 0])

polar_scatter = ax_polar.scatter(angles, radii, c=radii, cmap="viridis", alpha=0.6, edgecolors="none", s=20)
ax_polar.set_title("Polar Distribution of Continuous Self-Loop Actions (Dims 0 & 1)", fontsize=11, fontweight="bold", pad=15)
polar_colorbar = fig.colorbar(polar_scatter, ax=ax_polar, pad=0.1)
polar_colorbar.set_label("Action Vector Magnitude ||a||_2", fontsize=9)

plt.tight_layout()
plt.show()